# Generate_Data_v4 -- GP+Sigmoid Hybrid Generation Model

**Project**: GP-SPE Experimental Design Optimization  
**Core**: Sigmoid Theoretical Prior + GP Empirical Residual -> DDM Param Generator

---
## Workflow
| Step | Content | Output |
|------|---------|--------|
| 1. Setup | Import, locate root | -- |
| 2. Sigmoid | Define prior functions | -- |
| 3. Load | Read HDDM params | -- |
| **4. Calibrate** | Differential evolution | CSV |
| **5. Model** | Train 5 GPs | .pkl |
| **6. Evaluate** | In-sample fit | CSV+figs |
| **7. LOCV** | Leave-one-condition-out | CSV+figs |
| **8. External** | GP uncertainty -> new points | CSV+figs |

**Prequisites**: `numpy, pandas, matplotlib, scipy, scikit-learn`; HDDM fit done.


---
## Cell 1: Environment Setup & Path Config
Auto-locate project root via AGENTS.md. All paths relative.


In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
import pickle, warnings
warnings.filterwarnings("ignore")

BASE_DIR = Path.cwd()
while not (BASE_DIR / "AGENTS.md").exists() and BASE_DIR.parent != BASE_DIR:
    BASE_DIR = BASE_DIR.parent
if not (BASE_DIR / "AGENTS.md").exists():
    BASE_DIR = Path.cwd().resolve()
    for _ in range(5):
        if (BASE_DIR / "AGENTS.md").exists(): break
        BASE_DIR = BASE_DIR.parent

print(f"Project Root: {BASE_DIR}")
print(f"AGENTS.md exists: {(BASE_DIR / 'AGENTS.md').exists()}")

HDDM_PARAMS_PATH = BASE_DIR / "2_Data" / "Real_Data" / "HDDM_Traces" / "all_groups_ddm_params.csv"
OUT_GP_SIGMOID_DIR = BASE_DIR / "2_Data" / "Generate_Data" / "GP_Sigmoid"
OUT_CV_DIR = BASE_DIR / "2_Data" / "Generate_Data" / "Cross_Validation"
OUT_EV_DIR = BASE_DIR / "2_Data" / "Generate_Data" / "External_Verification"
FIG_GP_SIGMOID_DIR = BASE_DIR / "3_Figures" / "GP_Sigmoid"
FIG_CV_DIR = BASE_DIR / "3_Figures" / "Cross_Validation"
FIG_EV_DIR = BASE_DIR / "3_Figures" / "External_Verification"

for d in [OUT_GP_SIGMOID_DIR,OUT_CV_DIR,OUT_EV_DIR,FIG_GP_SIGMOID_DIR,FIG_CV_DIR,FIG_EV_DIR]:
    d.mkdir(parents=True, exist_ok=True)
plt.rcParams["font.sans-serif"] = ["SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

print(f"Input: {HDDM_PARAMS_PATH} -> exists: {HDDM_PARAMS_PATH.exists()}")
print(f"Output: {OUT_GP_SIGMOID_DIR}")
print("Setup complete!")


Project Root: d:\GitHub_programe\GitHub\Guassion-Process-Experiment-Design
AGENTS.md exists: True
Input: d:\GitHub_programe\GitHub\Guassion-Process-Experiment-Design\2_Data\Real_Data\HDDM_Traces\all_groups_ddm_params.csv -> exists: True
Output: d:\GitHub_programe\GitHub\Guassion-Process-Experiment-Design\2_Data\Generate_Data\GP_Sigmoid
Setup complete!


---
## Cell 2: Sigmoid Theoretical Prior Functions
S2 functions: (P,T,W) -> DDMs (v,a).


In [3]:
from scipy.special import expit as sigmoid

def v_P_Function(P, P1=4, k_min=0.1, k_max=0.05, gamma=0.2, P0=32):
    P = np.asarray(P, dtype=float)
    k = k_min + (k_max - k_min) / (1 + np.exp(-gamma * (P - P0)))
    return 1.0 / (1.0 + np.exp(-k * (P - P1)))

def compute_v_s2(T, P, condition_key, alaph1=1.5, alaph2=-0.4, gamma=0.2, T_0=100.0, k_T=0.01, base_scale=3.0):
    T = np.asarray(T, dtype=float); P = np.asarray(P, dtype=float)
    v_T = 1.0 / (1.0 + np.exp(-k_T * (T - T_0)))
    v_P = v_P_Function(P=P, P1=4, k_min=0.1, k_max=0.05, gamma=gamma, P0=32)
    v_0 = v_T * v_P * base_scale
    return np.where(np.asarray(condition_key) == 1, v_0 * (1 + alaph1), v_0 * (1 + alaph2))

def compute_a_s2(M, beta1=0.2, beta2=0.0, k=0.01, M_0=600.0, base_scale=3.0):
    M = np.asarray(M, dtype=float)
    a_0 = 1.0 / (1.0 + np.exp(-k * (M - M_0))) * base_scale
    return np.where(M > M_0, a_0 * (1 + beta1), a_0 * (1 + beta2))

print("Sigmoid functions defined!")


Sigmoid functions defined!


---
## Cell 3: Load Real HDDM Parameters
Read all_groups_ddm_params.csv.


In [4]:
try:
    df_real = pd.read_csv(HDDM_PARAMS_PATH)
except FileNotFoundError:
    print("ERROR: HDDM params not found. Run HDDM fitting first."); raise
print(f"Loaded {len(df_real)} conditions:\n")
disp_cols = ["group_id","P","T_ms","W_ms","M_ms","v_self_mean","v_stranger_mean","a_mean","t_mean","SPE_v"]
print(df_real[disp_cols].round(3).to_string(index=False))
print(f"\nRanges: v_self=[{df_real.v_self_mean.min():.2f},{df_real.v_self_mean.max():.2f}], "
      f"v_stranger=[{df_real.v_stranger_mean.min():.2f},{df_real.v_stranger_mean.max():.2f}], "
      f"a=[{df_real.a_mean.min():.2f},{df_real.a_mean.max():.2f}]")


Loaded 8 conditions:

 group_id   P  T_ms  W_ms  M_ms  v_self_mean  v_stranger_mean  a_mean  t_mean  SPE_v
        1   0    30   300   330       -3.385           -3.362   2.015   0.263 -0.023
        2   0    30   600   630       -2.531           -2.620   1.180   0.436  0.089
        3 120    30   600   630       -1.893           -1.733   1.218   0.348 -0.159
        4 120    80   600   680        0.639           -0.018   1.419   0.353  0.657
        5   8   100  1100  1200        1.349            0.635   1.335   0.398  0.714
        6 120   500  1500  2000        1.812            1.166   1.477   0.663  0.646
        7 120    80   800   880        1.205            0.753   1.092   0.405  0.452
        8 120    80   800   880        2.815            2.554   2.416   0.352  0.260

Ranges: v_self=[-3.39,2.81], v_stranger=[-3.36,2.55], a=[1.09,2.42]


---
## Cell 4: Sigmoid Calibration (Differential Evolution)
Optimize 7 params. Takes ~30-120s.


In [5]:
from scipy.optimize import differential_evolution
print("Starting Sigmoid calibration...")

def calib_obj(params, df_data):
    a1,a2,b1,b2,g,bsv,bsa = params
    e_vs=0.0; e_vo=0.0; e_a=0.0; n=len(df_data)
    for _,r in df_data.iterrows():
        M=r["T_ms"]+r["W_ms"]
        vs=compute_v_s2(r["T_ms"],r["P"],1,alaph1=a1,alaph2=a2,gamma=g,base_scale=bsv)
        vo=compute_v_s2(r["T_ms"],r["P"],0,alaph1=a1,alaph2=a2,gamma=g,base_scale=bsv)
        aa=compute_a_s2(M,beta1=b1,beta2=b2,base_scale=bsa)
        e_vs+=(float(vs)-r["v_self_mean"])**2
        e_vo+=(float(vo)-r["v_stranger_mean"])**2
        e_a+=(float(aa)-r["a_mean"])**2
    return np.sqrt((e_vs+e_vo+e_a)/(3*n))

bounds=[(0.1,3.0),(-2.0,1.0),(-1.0,2.0),(-1.0,1.0),(0.01,1.0),(1.0,10.0),(1.0,10.0)]
result = differential_evolution(calib_obj, bounds, args=(df_real,), maxiter=500, seed=42, popsize=15, tol=1e-8)
alaph1_opt,alaph2_opt,beta1_opt,beta2_opt,gamma_opt,bs_v_opt,bs_a_opt = result.x

print(f"\n--- Calibration Results ---")
print(f"alaph1={alaph1_opt:+.4f} alaph2={alaph2_opt:+.4f} gamma={gamma_opt:.4f}")
print(f"beta1={beta1_opt:+.4f} beta2={beta2_opt:+.4f} bsv={bs_v_opt:.4f} bsa={bs_a_opt:.4f}")
print(f"RMSE={result.fun:.6f} Converged={result.success}")

calibrated_params={"alaph1":alaph1_opt,"alaph2":alaph2_opt,"beta1":beta1_opt,"beta2":beta2_opt,
    "gamma":gamma_opt,"base_scale_v":bs_v_opt,"base_scale_a":bs_a_opt,
    "final_rmse":result.fun,"converged":result.success}
pd.DataFrame([calibrated_params]).to_csv(OUT_GP_SIGMOID_DIR/"sigmoid_calibrated_params.csv",index=False)
print("Saved calibrated params.")


Starting Sigmoid calibration...

--- Calibration Results ---
alaph1=+0.5487 alaph2=-0.2199 gamma=0.7013
beta1=-0.8274 beta2=+1.0000 bsv=1.0257 bsa=10.0000
RMSE=1.613868 Converged=True
Saved calibrated params.


---
## Cell 5: GP+Sigmoid Hybrid Model Class
5 independent GPs learn residual = real_ddm - sigmoid_pred.
Prediction = Sigmoid(theory) + GP(empirical residual)


In [6]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel

class GPSigmoidHybridModel:
    def __init__(self, seed=42):
        self.seed = seed
        bk = ConstantKernel(1.0) * RBF(length_scale=1.0) + WhiteKernel(noise_level=0.1)
        self.gp_v_self = GaussianProcessRegressor(kernel=bk, normalize_y=True, n_restarts_optimizer=10, random_state=seed)
        self.gp_v_stranger = GaussianProcessRegressor(kernel=bk, normalize_y=True, n_restarts_optimizer=10, random_state=seed+1)
        self.gp_a = GaussianProcessRegressor(kernel=bk, normalize_y=True, n_restarts_optimizer=10, random_state=seed+2)
        self.gp_t = GaussianProcessRegressor(kernel=bk, normalize_y=True, n_restarts_optimizer=10, random_state=seed+3)
        self.gp_z = GaussianProcessRegressor(kernel=bk, normalize_y=True, n_restarts_optimizer=10, random_state=seed+4)
        self._fitted=False; self._X_train=None; self._calib_params=None

    @staticmethod
    def normalize_PTW(P, T, W):
        Pn=(np.asarray(P,dtype=float)-60)/60
        Tn=(np.asarray(T,dtype=float)-265)/235
        Wn=(np.asarray(W,dtype=float)-900)/600
        return np.column_stack([Pn,Tn,Wn])

    @staticmethod
    def sigmoid_v_pred(T, P, cond, cp):
        T=np.asarray(T,float); P=np.asarray(P,float)
        def vp(Pv):
            k=0.1+(0.05-0.1)/(1+np.exp(-cp.get("gamma",0.2)*(Pv-32)))
            return 1.0/(1.0+np.exp(-k*(Pv-4)))
        vT=1.0/(1.0+np.exp(-0.01*(T-100.0)))
        v0=vT*vp(P)*cp.get("base_scale_v",3.0)
        cond=np.asarray(cond)
        return np.where(cond==1, v0*(1+cp.get("alaph1",1.5)), v0*(1+cp.get("alaph2",-0.4)))

    @staticmethod
    def sigmoid_a_pred(M, cp):
        M=np.asarray(M,float)
        a0=1.0/(1.0+np.exp(-0.01*(M-600.0)))*cp.get("base_scale_a",3.0)
        return np.where(M>600, a0*(1+cp.get("beta1",0.2)), a0*(1+cp.get("beta2",0.0)))

    def fit(self, df_data, calib_params=None):
        if calib_params is None:
            calib_params={"alaph1":1.5,"alaph2":-0.4,"beta1":0.2,"beta2":0.0,"gamma":0.2,"base_scale_v":3.0,"base_scale_a":3.0}
        self._calib_params=calib_params
        X=self.normalize_PTW(df_data["P"].values,df_data["T_ms"].values,df_data["W_ms"].values)
        self._X_train=X; n=len(df_data); Mv=df_data["T_ms"].values+df_data["W_ms"].values
        vs_real=df_data["v_self_mean"].values; vo_real=df_data["v_stranger_mean"].values
        a_real=df_data["a_mean"].values; t_real=df_data["t_mean"].values; z_real=df_data["z_mean"].values
        vs_sig=self.sigmoid_v_pred(df_data["T_ms"].values,df_data["P"].values,np.ones(n),calib_params)
        vo_sig=self.sigmoid_v_pred(df_data["T_ms"].values,df_data["P"].values,np.zeros(n),calib_params)
        a_sig=self.sigmoid_a_pred(Mv,calib_params)
        self.gp_v_self.fit(X, vs_real-vs_sig)
        self.gp_v_stranger.fit(X, vo_real-vo_sig)
        self.gp_a.fit(X, a_real-a_sig)
        self.gp_t.fit(X, t_real-np.full(n,np.mean(t_real)))
        self.gp_z.fit(X, z_real-np.full(n,np.mean(z_real)))
        self._fitted=True; return self

    def predict(self, P, T, W, condition_key, return_std=False):
        if not self._fitted: raise RuntimeError("Model not fitted")
        P=np.atleast_1d(np.asarray(P,float)); T=np.atleast_1d(np.asarray(T,float))
        W=np.atleast_1d(np.asarray(W,float)); cond=np.atleast_1d(np.asarray(condition_key))
        n=len(P); X=self.normalize_PTW(P,T,W)
        vs_s=self.sigmoid_v_pred(T,P,np.ones(n),self._calib_params)
        vo_s=self.sigmoid_v_pred(T,P,np.zeros(n),self._calib_params)
        a_s=self.sigmoid_a_pred(T+W,self._calib_params)
        t_s=np.full(n,self._calib_params.get("t_baseline",0.35))
        z_s=np.full(n,self._calib_params.get("z_baseline",0.55))
        vr_self,vr_self_std=self.gp_v_self.predict(X,return_std=True)
        vr_other,vr_other_std=self.gp_v_stranger.predict(X,return_std=True)
        a_r,a_std=self.gp_a.predict(X,return_std=True)
        t_r,t_std=self.gp_t.predict(X,return_std=True)
        z_r,z_std=self.gp_z.predict(X,return_std=True)
        v=np.where(cond==1, vs_s+vr_self, vo_s+vr_other)
        v_std=np.where(cond==1, vr_self_std, vr_other_std)
        a=np.maximum(a_s+a_r,0.1); t=np.maximum(t_s+t_r,0.01)
        z=np.clip(z_s+z_r,0.01,a-0.01)
        if return_std: return ((v,a,t,z),(v_std,a_std,t_std,z_std))
        return (v,a,t,z)

    def predict_params_full(self, P, T, W):
        vs,as_,ts,zs = self.predict(P,T,W,1)
        vo,ao,to,zo = self.predict(P,T,W,0)
        return vs,vo,as_,ts,zs

    def save(self, path):
        with open(path,"wb") as f:
            pickle.dump({"seed":self.seed,"_fitted":self._fitted,"_X_train":self._X_train,
                "_calib_params":self._calib_params,"gp_v_self":self.gp_v_self,
                "gp_v_stranger":self.gp_v_stranger,"gp_a":self.gp_a,"gp_t":self.gp_t,"gp_z":self.gp_z},f)

    @classmethod
    def load(cls, path):
        with open(path,"rb") as f: d=pickle.load(f)
        m=cls(seed=d["seed"]); m._fitted=d["_fitted"]; m._X_train=d["_X_train"]
        m._calib_params=d["_calib_params"]; m.gp_v_self=d["gp_v_self"]
        m.gp_v_stranger=d["gp_v_stranger"]; m.gp_a=d["gp_a"]; m.gp_t=d["gp_t"]; m.gp_z=d["gp_z"]
        return m

print("GPSigmoidHybridModel defined! 5 GPs: v_self, v_stranger, a, t, z")


GPSigmoidHybridModel defined! 5 GPs: v_self, v_stranger, a, t, z


---
## Cell 6: Train GP+Sigmoid Hybrid Model
Fit 5 GPs on 8 conditions.


In [7]:
print("Training GP+Sigmoid hybrid model...")
calib_params_with_baseline = dict(calibrated_params)
calib_params_with_baseline["t_baseline"] = np.mean(df_real["t_mean"])
calib_params_with_baseline["z_baseline"] = np.mean(df_real["z_mean"])
model = GPSigmoidHybridModel(seed=42)
model.fit(df_real, calib_params=calib_params_with_baseline)
print("Model trained!")
print(f"GP v_self kernel: {model.gp_v_self.kernel_}")
print(f"GP a kernel:      {model.gp_a.kernel_}")
model_path = OUT_GP_SIGMOID_DIR / "gp_sigmoid_hybrid_model.pkl"
model.save(model_path)
print(f"Saved: {model_path}")


Training GP+Sigmoid hybrid model...
Model trained!
GP v_self kernel: 0.8**2 * RBF(length_scale=0.011) + WhiteKernel(noise_level=0.342)
GP a kernel:      0.00316**2 * RBF(length_scale=0.396) + WhiteKernel(noise_level=1)
Saved: d:\GitHub_programe\GitHub\Guassion-Process-Experiment-Design\2_Data\Generate_Data\GP_Sigmoid\gp_sigmoid_hybrid_model.pkl


---
## Cell 7: Training Set Fit Evaluation
Predict on 8 training conditions, compute RMSE and r.


In [8]:
print("Evaluating training fit...")
predictions = []
for _, row in df_real.iterrows():
    vs,vo,a,t_,z = model.predict_params_full(row["P"],row["T_ms"],row["W_ms"])
    vs=np.asarray(vs).item(); vo=np.asarray(vo).item()
    a=np.asarray(a).item(); t_=np.asarray(t_).item(); z=np.asarray(z).item()
    predictions.append({"group_id":row["group_id"],"P":row["P"],"T_ms":row["T_ms"],"W_ms":row["W_ms"],
        "v_self_real":row["v_self_mean"],"v_self_pred":vs,
        "v_stranger_real":row["v_stranger_mean"],"v_stranger_pred":vo,
        "a_real":row["a_mean"],"a_pred":a,"t_real":row["t_mean"],"t_pred":t_,
        "z_real":row["z_mean"],"z_pred":z,
        "SPE_v_real":row["v_self_mean"]-row["v_stranger_mean"],"SPE_v_pred":vs-vo})
df_pred = pd.DataFrame(predictions)
df_pred.to_csv(OUT_GP_SIGMOID_DIR/"gp_sigmoid_predictions.csv",index=False)
print("\nIn-sample fit:")
for param in ["v_self","v_stranger","a","t","z","SPE_v"]:
    rv=df_pred[f"{param}_real"]; pv=df_pred[f"{param}_pred"]
    rmse=np.sqrt(np.mean((rv-pv)**2)); corr=np.corrcoef(rv,pv)[0,1]
    print(f"  {param:12s}: RMSE={rmse:.4f}, r={corr:.3f}")
    if param=="SPE_v": spe_rmse=rmse; spe_corr=corr
print("Saved predictions.")


Evaluating training fit...

In-sample fit:
  v_self      : RMSE=0.7095, r=0.980
  v_stranger  : RMSE=0.7464, r=0.962
  a           : RMSE=0.4611, r=0.240
  t           : RMSE=0.0163, r=0.992
  z           : RMSE=0.0003, r=1.000
  SPE_v       : RMSE=0.1722, r=0.873
Saved predictions.


---
## Cell 8: Visualization - Real vs Predicted


In [9]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for i, (param, ax, yl) in enumerate([
    ("v_self",axes[0,0],"v (Self)"),("v_stranger",axes[0,1],"v (Stranger)"),
    ("a",axes[0,2],"a"),("t",axes[1,0],"t (s)"),("z",axes[1,1],"z"),("SPE_v",axes[1,2],"SPE_v")]):
    rv=df_pred[f"{param}_real"]; pv=df_pred[f"{param}_pred"]
    rmse=np.sqrt(np.mean((rv-pv)**2)); corr=np.corrcoef(rv,pv)[0,1]
    ax.scatter(rv,pv,s=80,c="steelblue",edgecolors="black",zorder=5)
    for j,(_,r) in enumerate(df_pred.iterrows()):
        ax.annotate(f"G{r['group_id']:.0f}",(rv.iloc[j],pv.iloc[j]),textcoords="offset points",xytext=(5,5),fontsize=7)
    av=np.concatenate([rv.values,pv.values])
    lo=min(av)*1.3 if min(av)<0 else min(av)*0.7
    hi=max(av)*1.3 if max(av)>0 else max(av)*0.7
    ax.plot([lo,hi],[lo,hi],"r--",alpha=0.5,label="y=x")
    ax.set_xlim(lo,hi); ax.set_ylim(lo,hi)
    ax.set_xlabel("Real HDDM"); ax.set_ylabel("GP+Sigmoid")
    ax.set_title(f"{yl} (RMSE={rmse:.3f}, r={corr:.3f})"); ax.legend(fontsize=7)
plt.suptitle("GP+Sigmoid: Real vs Predicted DDM Parameters",fontsize=14,y=1.01)
plt.tight_layout(); plt.savefig(FIG_GP_SIGMOID_DIR/"gp_sigmoid_real_vs_pred.png",dpi=200,bbox_inches="tight")
plt.close()

# SPE bar chart
fig2,ax2=plt.subplots(figsize=(8,5))
labels=[f"G{r['group_id']:.0f}" for _,r in df_pred.iterrows()]
x=np.arange(len(labels)); w=0.35
ax2.bar(x-w/2,df_pred["SPE_v_real"],w,label="Real HDDM",color="steelblue",alpha=0.8)
ax2.bar(x+w/2,df_pred["SPE_v_pred"],w,label="GP+Sigmoid",color="coral",alpha=0.8)
ax2.set_xticks(x); ax2.set_xticklabels(labels,rotation=45)
ax2.set_ylabel("SPE_v"); ax2.set_title(f"SPE_v: Real vs GP+Sigmoid (RMSE={spe_rmse:.3f})")
ax2.legend(); plt.tight_layout()
plt.savefig(FIG_GP_SIGMOID_DIR/"spe_v_comparison.png",dpi=200,bbox_inches="tight")
plt.close()
print(f"Figs saved: {FIG_GP_SIGMOID_DIR}")


Figs saved: d:\GitHub_programe\GitHub\Guassion-Process-Experiment-Design\3_Figures\GP_Sigmoid


---
## Cell 9: LOCV Cross-Validation
Hold out one condition at a time. Test generalization.


In [10]:
print("="*60)
print("Leave-One-Condition-Out Cross-Validation")
print("="*60)
group_ids = sorted(df_real["group_id"].unique())
print(f"Conditions: {[int(g) for g in group_ids]}\n")
all_results = []
for held_out in group_ids:
    print(f"Holding out Group {int(held_out)}")
    df_tr = df_real[df_real["group_id"]!=held_out].copy()
    df_te = df_real[df_real["group_id"]==held_out].copy()
    cv_m = GPSigmoidHybridModel(seed=42)
    cv_m.fit(df_tr, calib_params=calib_params_with_baseline)
    r = df_te.iloc[0]
    vs,vo,a,t_,z = cv_m.predict_params_full(r["P"],r["T_ms"],r["W_ms"])
    vs=np.asarray(vs).item(); vo=np.asarray(vo).item()
    a=np.asarray(a).item(); t_=np.asarray(t_).item(); z=np.asarray(z).item()
    res={"held_out_group":int(held_out),"P":r["P"],"T_ms":r["T_ms"],"W_ms":r["W_ms"],
        "v_self_real":r["v_self_mean"],"v_self_pred":vs,
        "v_stranger_real":r["v_stranger_mean"],"v_stranger_pred":vo,
        "a_real":r["a_mean"],"a_pred":a,"t_real":r["t_mean"],"t_pred":t_,
        "z_real":r["z_mean"],"z_pred":z}
    res["SPE_v_real"]=res["v_self_real"]-res["v_stranger_real"]
    res["SPE_v_pred"]=res["v_self_pred"]-res["v_stranger_pred"]
    for p in ["v_self","v_stranger","a","t","z","SPE_v"]:
        res[f"{p}_error"]=res[f"{p}_real"]-res[f"{p}_pred"]
    all_results.append(res)
    print(f"  SPE_v: real={res["SPE_v_real"]:.3f}, pred={res["SPE_v_pred"]:.3f}, err={res["SPE_v_error"]:.3f}")
df_locv = pd.DataFrame(all_results)
df_locv.to_csv(OUT_CV_DIR/"locv_results.csv",index=False)
print("\nLOCV Metrics:")
locv_metrics = {}
for param in ["v_self","v_stranger","a","t","z","SPE_v"]:
    rv=df_locv[f"{param}_real"]; pv=df_locv[f"{param}_pred"]
    rmse=np.sqrt(np.mean((rv-pv)**2)); mae=np.mean(np.abs(rv-pv))
    try: corr=np.corrcoef(rv,pv)[0,1]
    except: corr=np.nan
    locv_metrics[param]={"RMSE":rmse,"MAE":mae,"r":corr}
    print(f"  {param:12s}: RMSE={rmse:.4f}, MAE={mae:.4f}, r={corr:.3f}")
pd.DataFrame(locv_metrics).T.to_csv(OUT_CV_DIR/"locv_metrics.csv")
print(f"\nSaved: {OUT_CV_DIR}")


Leave-One-Condition-Out Cross-Validation
Conditions: [1, 2, 3, 4, 5, 6, 7, 8]

Holding out Group 1
  SPE_v: real=-0.023, pred=0.137, err=-0.160
Holding out Group 2
  SPE_v: real=0.089, pred=0.121, err=-0.032
Holding out Group 3
  SPE_v: real=-0.159, pred=0.379, err=-0.538
Holding out Group 4
  SPE_v: real=0.657, pred=0.324, err=0.333
Holding out Group 5
  SPE_v: real=0.714, pred=0.219, err=0.495
Holding out Group 6
  SPE_v: real=0.646, pred=0.804, err=-0.158
Holding out Group 7
  SPE_v: real=0.452, pred=-2.639, err=3.091
Holding out Group 8
  SPE_v: real=0.260, pred=-0.070, err=0.330

LOCV Metrics:
  v_self      : RMSE=2.1829, MAE=1.9889, r=-0.019
  v_stranger  : RMSE=2.0225, MAE=1.8366, r=0.207
  a           : RMSE=0.7570, MAE=0.5854, r=-0.425
  t           : RMSE=0.1239, MAE=0.0933, r=-0.040
  z           : RMSE=0.0933, MAE=0.0692, r=0.368
  SPE_v       : RMSE=1.1381, MAE=0.6421, r=-0.055

Saved: d:\GitHub_programe\GitHub\Guassion-Process-Experiment-Design\2_Data\Generate_Data\Cross_

---
## Cell 10: LOCV Visualization


In [11]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for i, (param, ax, yl) in enumerate([
    ("v_self",axes[0,0],"v (Self)"),("v_stranger",axes[0,1],"v (Stranger)"),
    ("a",axes[0,2],"a"),("t",axes[1,0],"t (s)"),("z",axes[1,1],"z"),("SPE_v",axes[1,2],"SPE_v")]):
    rv=df_locv[f"{param}_real"]; pv=df_locv[f"{param}_pred"]
    rmse=np.sqrt(np.mean((rv-pv)**2)); corr=np.corrcoef(rv,pv)[0,1]
    ax.scatter(rv,pv,s=100,c="steelblue",edgecolors="black",zorder=5)
    for j,(_,r) in enumerate(df_locv.iterrows()):
        ax.annotate(f"G{r["held_out_group"]:.0f}",(rv.iloc[j],pv.iloc[j]),textcoords="offset points",xytext=(5,5),fontsize=8)
    av=np.concatenate([rv.values,pv.values])
    lo=min(av)*1.3 if min(av)<0 else min(av)*0.7
    hi=max(av)*1.3 if max(av)>0 else max(av)*0.7
    lo=min(lo,hi-0.1); hi=max(hi,lo+0.1)
    ax.plot([lo,hi],[lo,hi],"r--",alpha=0.5,label="y=x")
    ax.set_xlim(lo,hi); ax.set_ylim(lo,hi)
    ax.set_xlabel("Real HDDM"); ax.set_ylabel("LOCV Predicted")
    ax.set_title(f"{yl} (RMSE={rmse:.3f}, r={corr:.3f})"); ax.legend(fontsize=7)
plt.suptitle("LOCV: Real vs Predicted (8 folds)",fontsize=14,y=1.01)
plt.tight_layout(); plt.savefig(FIG_CV_DIR/"locv_scatter.png",dpi=200,bbox_inches="tight"); plt.close()

fig2,ax2=plt.subplots(figsize=(8,6))
params=list(locv_metrics.keys()); rmses=[locv_metrics[p]["RMSE"] for p in params]
colors=plt.cm.viridis(np.linspace(0.2,0.8,len(params)))
bars=ax2.bar(params,rmses,color=colors,edgecolor="black")
for b,v in zip(bars,rmses):
    ax2.text(b.get_x()+b.get_width()/2,b.get_height()+0.01,f"{v:.3f}",ha="center",va="bottom",fontsize=9)
ax2.set_ylabel("RMSE"); ax2.set_title("LOCV RMSE by Parameter")
plt.tight_layout(); plt.savefig(FIG_CV_DIR/"locv_rmse.png",dpi=200,bbox_inches="tight"); plt.close()
print(f"Figs saved: {FIG_CV_DIR}")


Figs saved: d:\GitHub_programe\GitHub\Guassion-Process-Experiment-Design\3_Figures\Cross_Validation


---
## Cell 11: External Verification - Find Optimal New Design Points
Sample (P,T,W) on dense grid. Find points with highest GP uncertainty.


In [12]:
print("="*60)
print("External Verification: GP Uncertainty Surface")
print("="*60)
Pr=np.linspace(0,120,15); Tr=np.linspace(30,500,15); Wr=np.linspace(300,1500,15)
Pg,Tg,Wg=np.meshgrid(Pr,Tr,Wr,indexing="ij")
Pf=Pg.ravel(); Tf=Tg.ravel(); Wf=Wg.ravel(); npt=len(Pf)
print(f"Grid: {len(Pr)}x{len(Tr)}x{len(Wr)} = {npt} points")
all_unc=np.zeros(npt)
for s in range(0,npt,500):
    e=min(s+500,npt)
    Xb=model.normalize_PTW(Pf[s:e],Tf[s:e],Wf[s:e])
    _,vs=model.gp_v_self.predict(Xb,return_std=True)
    _,as_=model.gp_a.predict(Xb,return_std=True)
    _,ts=model.gp_t.predict(Xb,return_std=True)
    _,zs=model.gp_z.predict(Xb,return_std=True)
    all_unc[s:e]=vs+as_+ts+zs
print(f"Uncertainty range: {all_unc.min():.4f} - {all_unc.max():.4f}")

# Exclude near existing
ep=df_real[["P","T_ms","W_ms"]].values
en=model.normalize_PTW(ep[:,0],ep[:,1],ep[:,2])
an=model.normalize_PTW(Pf,Tf,Wf)
tc=np.zeros(npt,dtype=bool)
for epp in en:
    d=np.sqrt(np.sum((an-epp)**2,axis=1))
    tc|=(d<0.15)
mu=all_unc.copy(); mu[tc]=-np.inf
topn=12; topi=np.argsort(mu)[::-1][:topn]

cands=[{"P":int(Pf[i]),"T_ms":int(Tf[i]),"W_ms":int(Wf[i]),"M_ms":int(Tf[i]+Wf[i]),"total_unc":all_unc[i]} for i in topi]
dfc=pd.DataFrame(cands); dfc["rank"]=range(1,len(dfc)+1)
dfc=dfc[["rank","P","T_ms","W_ms","M_ms","total_unc"]]
dfc.to_csv(OUT_EV_DIR/"optimal_design_points.csv",index=False)
print(f"\nTop candidates:")
for _,r in dfc.head(8).iterrows():
    print(f"  #{r['rank']:.0f}: P={r['P']:.0f}, T={r['T_ms']:.0f}ms, W={r['W_ms']:.0f}ms -> unc={r['total_unc']:.4f}")
print(f"\nSaved: {OUT_EV_DIR}")


External Verification: GP Uncertainty Surface
Grid: 15x15x15 = 3375 points
Uncertainty range: 1.9268 - 2.5825

Top candidates:
  #1: P=42, T=500ms, W=300ms -> unc=2.5825
  #2: P=51, T=500ms, W=300ms -> unc=2.5825
  #3: P=42, T=500ms, W=385ms -> unc=2.5825
  #4: P=34, T=500ms, W=300ms -> unc=2.5825
  #5: P=34, T=500ms, W=385ms -> unc=2.5825
  #6: P=51, T=500ms, W=385ms -> unc=2.5825
  #7: P=25, T=500ms, W=300ms -> unc=2.5825
  #8: P=60, T=500ms, W=300ms -> unc=2.5825

Saved: d:\GitHub_programe\GitHub\Guassion-Process-Experiment-Design\2_Data\Generate_Data\External_Verification


---
## Cell 12: Visualization - GP Uncertainty Surface


In [13]:
fig,axes=plt.subplots(1,2,figsize=(16,6))
wt=np.median(Wr); wm=np.abs(Wf-wt)<60
ax=axes[0]
sc=ax.scatter(Pf[wm],Tf[wm],c=all_unc[wm],cmap="viridis",s=30,alpha=0.7)
plt.colorbar(sc,ax=ax,label="Total Uncertainty")
ax.scatter(ep[:,0],ep[:,1],c="red",marker="X",s=100,edgecolors="black",linewidths=1.5,label="Existing")
for i,(_,r) in enumerate(df_real.iterrows()):
    ax.annotate(f"G{r['group_id']:.0f}",(r["P"],r["T_ms"]),textcoords="offset points",xytext=(5,5),fontsize=7,color="red")
for _,r in dfc.head(4).iterrows():
    ax.scatter(r["P"],r["T_ms"],c="cyan",marker="D",s=80,edgecolors="black",linewidths=1.5,zorder=11)
    ax.annotate(f"#{r['rank']:.0f}",(r["P"],r["T_ms"]),textcoords="offset points",xytext=(-10,-10),fontsize=8,color="cyan")
ax.set_xlabel("P"); ax.set_ylabel("T (ms)")
ax.set_title(f"GP Uncertainty in (P,T) space at W~{wt:.0f}ms"); ax.legend(fontsize=7)

ax=axes[1]
sorted_c=dfc.sort_values("total_unc",ascending=True)
labels_c=[f"#{r['rank']:.0f}: P={r['P']:.0f},T={r['T_ms']:.0f},W={r['W_ms']:.0f}" for _,r in sorted_c.iterrows()]
colors_c=plt.cm.viridis(np.linspace(0.2,0.9,len(sorted_c)))
ax.barh(labels_c,sorted_c["total_unc"].values,color=colors_c,edgecolor="black")
ax.set_xlabel("Total GP Uncertainty")
ax.set_title("Top Candidate Design Points")
plt.tight_layout(); plt.savefig(FIG_EV_DIR/"uncertainty_surface.png",dpi=200,bbox_inches="tight"); plt.close()
print(f"Figs saved: {FIG_EV_DIR}")


Figs saved: d:\GitHub_programe\GitHub\Guassion-Process-Experiment-Design\3_Figures\External_Verification


---
## Summary: Output Files

| Directory | File | Content |
|-----------|------|---------|
| `2_Data/Generate_Data/GP_Sigmoid/` | `sigmoid_calibrated_params.csv` | Calibrated Sigmoid params |
| | `gp_sigmoid_predictions.csv` | Training set predictions |
| | `gp_sigmoid_hybrid_model.pkl` | Trained GP+Sigmoid model |
| `2_Data/Generate_Data/Cross_Validation/` | `locv_results.csv` | LOCV per-condition results |
| | `locv_metrics.csv` | LOCV summary metrics |
| `2_Data/Generate_Data/External_Verification/` | `optimal_design_points.csv` | Candidate new design points |
| `3_Figures/GP_Sigmoid/` | `gp_sigmoid_real_vs_pred.png` | Training fit scatter |
| | `spe_v_comparison.png` | SPE comparison bar |
| `3_Figures/Cross_Validation/` | `locv_scatter.png` | LOCV scatter |
| | `locv_rmse.png` | LOCV RMSE bar |
| `3_Figures/External_Verification/` | `uncertainty_surface.png` | GP uncertainty map |

---
## Next Steps

1. Verify training fit metrics (particularly SPE_v) are satisfactory
2. Review LOCV results to understand generalization limits
3. Use candidate design points for new data collection
4. Re-train model with new data to improve predictions

---
*Notebook v4. Built for GP-SPE project.*
